# TIES-Merging — what was merged, and how

**Question.** When we merge the top-K epoch checkpoints of a run (or several runs), what do the
weight updates look like, what does TIES-Merging ([Yadav et al. 2023](https://arxiv.org/abs/2306.01708))
keep, where do the models disagree, and does any of it change the local WF1?

TIES in three steps, with task vectors $\tau_t = \theta_t - \theta_{\text{init}}$ against the pretrained
Qwen3-ASR backbone ($\theta_{\text{init}}$ = hub weights; the SLS head has no pretrained init and is plain-averaged):

1. **Trim** — keep the top-$k$% of each $\tau_t$ by magnitude (global over the task vector), zero the rest.
2. **Elect** — $\gamma = \operatorname{sign}\big(\sum_t \hat\tau_t\big)$.
3. **Disjoint merge** — $\tau_m^p = \frac{1}{|A^p|}\sum_{t \in A^p} \hat\tau_t^p$ with $A^p = \{t : \operatorname{sign}(\hat\tau_t^p) = \gamma^p\}$, then $\theta_m = \theta_{\text{init}} + \lambda\,\tau_m$.

**Inputs** (this notebook only reads cached aggregates, so it reruns in seconds and needs no GPU):
- `ties_stats.json` / `ties_stats.npz` from `scripts/ties_merge.py --stats` (or `python -m ties_merging.checks`), for one merge setting $(k, \lambda)$ plus a $k$ grid.
- optionally a `scripts/ties_eval.py` sweep directory (`summary.csv`) for the performance plots.

Every figure function lives in `SLS_setup/ties_merging/plots.py`; `python -m ties_merging.plots --stats <dir>` renders the same PNGs for a report.

In [ ]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "SLS_setup"))
from ties_merging import plots as P, stats as S

# --- inputs: a ties_merge.py --stats output dir and (optional) a ties_eval.py sweep dir -------------
RUN = REPO / "SLS_setup/exp/B2f_qwen17_rtc_musan_loud23_cos8_lr2.5e6_s1234_epoch8_bs16_20261009132623"
STATS_DIR = Path(os.environ.get("TIES_STATS_DIR", RUN / "ties_k20_lam1"))
SWEEP_DIR = Path(os.environ.get("TIES_SWEEP_DIR", RUN / "local_eval/ties_sweep"))

P.apply_style()
js, npz = S.load(STATS_DIR)
cfg = json.loads((STATS_DIR / "ties_config.json").read_text()) if (STATS_DIR / "ties_config.json").is_file() else {}
labels = P.model_labels(js)
print(f"{js['T']} ingredients, {js['n_params'] / 1e6:.1f}M TIES parameters, k = {js['k']:g}%, lambda = {js['lambda']:g}, "
      f"base = {js['base']['label']}")
for lab, m in zip(labels, js["models"]):
    print(f"  {lab:<10} ||tau|| = {m['norm']:.3f}   exact zeros = {m['n_exact_zero']}")

## 1. The ingredients: how far did each checkpoint move, and in the same direction?

$\|\tau_t\|$ per checkpoint, and the cosine between the task vectors. Checkpoints of one run move along almost
the same direction (cosine close to 1): the "tasks" being merged are not different tasks, which is the
regime the paper's sign-conflict analysis assumes. Cross-run lists sit lower.

In [ ]:
fig = P.fig_cosine(js, npz); plt.show()
pd.DataFrame({"ingredient": labels, "||tau||": [m["norm"] for m in js["models"]],
              "kept at k (%)": [100 * m["kept_frac"][str(float(js["k"]))] for m in js["models"]],
              "agree with elected sign (%)": [100 * m["elect_agree_frac"] for m in js["models"]],
              "share of merged entries (%)": [100 * m["contribution_frac"] for m in js["models"]]}).round(3)

## 2. Trimming: are the updates concentrated in a few parameters?

The paper's premise (its Fig. 3 / 10): a small fraction of high-magnitude entries carries the fine-tuning, so
keeping the top 20% loses nothing. The curve below is the share of $\sum\tau^2$ held by the top-$q$% entries;
the diagonal is what uniform magnitudes would give. AdamW updates of a full fine-tune are much flatter than the
paper's PEFT / ViT task vectors — read the number at the $k$ line.

In [ ]:
fig = P.fig_concentration(js, npz); plt.show()
fig = P.fig_magnitude_hist(js, npz); plt.show()
pd.DataFrame({lab: m["quantiles_abs"] for lab, m in zip(labels, js["models"])}).T.rename(
    columns=lambda q: f"|tau| p{float(q) * 100:g}")

## 3. Sign conflicts: do the checkpoints disagree, and how much does trimming leave to disagree on?

Paper Fig. 4 / 14 / 15: conflicts persist after trimming and grow with the number of models. Left: the fraction of
positions kept by $\ge 2$ models whose kept signs disagree, as a function of $k$. Right: the same at the merge's
$k$ as checkpoints are added in list order. For same-run checkpoints the conflict fraction below $k \approx 50$ is
essentially zero: **the Elect step is then a no-op and TIES reduces to "trim, then average the survivors".**

In [ ]:
fig = P.fig_conflicts(js, npz); plt.show()
pd.DataFrame(js["conflict_vs_k"]).T.rename_axis("k").round(5)

## 4. Interference: what the merged value looks like, TIES vs plain mean

Paper Fig. 9. Left (redundant-value interference): positions grouped by how many models keep them. A plain mean
divides every entry by $T$, so a position that only one model found important shrinks to $1/T$ of its value;
TIES divides by $|A^p|$ (the number of agreeing keepers) and gives it its full value. Right (sign interference):
positions kept by $\ge 2$ models grouped by their sign agreement; TIES averages the majority sign only.

In [ ]:
fig = P.fig_interference(js, npz); plt.show()
pd.DataFrame(js["kept_count_groups"]).T.rename_axis("kept by (models)").round(6)

## 5. Who wins the election, and where in the network?

Per ingredient: the share of its kept entries that match the elected sign, and its share of the merged entries.
Per layer group: the relative update norm, the kept fraction at $k$ (global trimming keeps more of the layers
with larger updates), and the merge's shared / conflict / merged fractions.

In [ ]:
fig = P.fig_election(js); plt.show()
fig = P.fig_layers(js); plt.show()

## 6. Does it change WF1? (needs a `scripts/ties_eval.py` sweep)

Paper Fig. 13 (performance vs $\lambda$ and vs $k$) and the ablation table (Fig. 12). Cells are selected on
**half A** and reported on **half B**; the dashed line is the plain mean of the same checkpoints
(`main_eval.merge_checkpoints`, the current production merge). Variants: `_notrim`, `_noelect`, `_meanT`
(sum / $T$ instead of / $|A^p|$) and `single<i>_k..` (one checkpoint, trimmed: the T = 1 trim curve).

In [ ]:
if (SWEEP_DIR / "summary.csv").is_file():
    df = P.load_sweep(SWEEP_DIR)
    display(P.ablation_table(df))
    if len(P._ties_cells(df)):
        for metric in ("half_A.wf1", "half_B.wf1"):
            if metric in df and df[metric].notna().any():
                fig = P.fig_perf_vs_lambda(df, metric); plt.show()
        fig = P.fig_perf_vs_k(df); plt.show()
else:
    print(f"no sweep at {SWEEP_DIR}: run scripts/ties_eval.py first (see SLS_setup/ties_merging/README.md)")

## Conclusions

Fill in after the sweep. Things to state explicitly:

- **Concentration.** What share of the update energy sits in the top 20%? (Section 2.) If it is far from the paper's
  "almost all", trimming removes real signal and $k$ should be swept upward.
- **Conflicts.** Is the conflict fraction at the merge's $k$ near zero? Then Elect does nothing here and the only
  active ingredients are trimming and the $1/|A^p|$ renormalisation (compare the `_meanT` cell).
- **Performance.** Does any $(k, \lambda)$ cell beat the plain mean on half B outside the $\pm 0.5$ WF1 band, with
  the expected signature (heldout / echo bonafide recall up, noisy EER down)? Confirm with
  `scripts/b3_compare_evals.py <mean cell> <ties cell> --ci 2000` before adopting.